In [1]:
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import os
import random
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import random
import numpy as np
import mlflow 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

# device = torch.device("cpu")  # Force CPU for testing time
random_seed = 42


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Z_prediction")
mlflow.enable_system_metrics_logging()

In [3]:
def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved {len(df)} rows to {save_path}")

In [4]:
""""
video_folder = "../data"
model_path = "../data/pose_landmarker.task"
output_folder = "../Jakob_movement_CSV"

video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

for file_name in os.listdir(video_folder):
    
    if not any(file_name.lower().endswith(ext) for ext in video_extensions):
        continue

    video_path = os.path.join(video_folder, file_name)

    base_name = os.path.splitext(file_name)[0]
    save_name = f"{base_name}_mediapipe.csv"
    save_path = os.path.join(output_folder, save_name)

    print(f"Processing: {file_name}")

    extract_mediapipe_to_csv(
        data_path=video_path,
        save_path=save_path,
        model_path=model_path
    )
print("Done.")
"""

'"\nvideo_folder = "../data"\nmodel_path = "../data/pose_landmarker.task"\noutput_folder = "../Jakob_movement_CSV"\n\nvideo_extensions = [".mp4", ".mov", ".avi", ".mkv"]\n\nfor file_name in os.listdir(video_folder):\n\n    if not any(file_name.lower().endswith(ext) for ext in video_extensions):\n        continue\n\n    video_path = os.path.join(video_folder, file_name)\n\n    base_name = os.path.splitext(file_name)[0]\n    save_name = f"{base_name}_mediapipe.csv"\n    save_path = os.path.join(output_folder, save_name)\n\n    print(f"Processing: {file_name}")\n\n    extract_mediapipe_to_csv(\n        data_path=video_path,\n        save_path=save_path,\n        model_path=model_path\n    )\nprint("Done.")\n'

## Compare mediapipe with irl measurments

In [5]:
# Scale factors
df = pd.read_csv("../Jakob_movement_CSV/mark-still_mediapipe.csv")
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}

x_scale = ground_truth["hip_to_hip"] / (df["left_hip_x"] - df["right_hip_x"]).abs().mean()
y_scale = ground_truth["hip_to_shoulder"] / (df["left_hip_y"] - df["left_shoulder_y"]).abs().mean()

print(f"X scale: {x_scale:.2f} cm per unit")
print(f"Y scale: {y_scale:.2f} cm per unit")

# Scale all coordinates
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * x_scale
    df[f"{joint}_y"] = df[f"{joint}_y"] * y_scale

# Now calculate Euclidean in cm
def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

df["hip_to_shoulder"]      = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]           = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]        = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"] = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                        df["right_shoulder_x"], df["right_shoulder_y"])
df["shoulder_to_shoulder"] = np.mean(df["left_shoulder_x"] - df["right_shoulder_x"])

avg = df[["hip_to_shoulder", "knee_to_hip", "hip_to_hip", 
          "knee_to_ankle", "shoulder_to_shoulder"]].mean()

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key]
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

X scale: 468.87 cm per unit
Y scale: 238.31 cm per unit
Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.7          0.7        1.5%
knee_to_hip                     44.0            41.1          2.9        6.6%
hip_to_hip                      25.0            25.0          0.0        0.0%
knee_to_ankle                   43.0            37.2          5.8       13.4%
shoulder_to_shoulder            38.0            40.8          2.8        7.3%


In [6]:
# Scale factors
df = pd.read_csv("../Jakob_movement_CSV/upphöjd-still_mediapipe.csv")
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}

x_scale = ground_truth["hip_to_hip"] / (df["left_hip_x"] - df["right_hip_x"]).abs().mean()
y_scale = ground_truth["hip_to_shoulder"] / (df["left_hip_y"] - df["left_shoulder_y"]).abs().mean()

print(f"X scale: {x_scale:.2f} cm per unit")
print(f"Y scale: {y_scale:.2f} cm per unit")

# Scale all coordinates
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * x_scale
    df[f"{joint}_y"] = df[f"{joint}_y"] * y_scale

# Now calculate Euclidean in cm
def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

df["hip_to_shoulder"]      = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]           = euclidean(df["left_hip_x"], df["left_hip_y"],
                                        df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]        = euclidean(df["left_knee_x"], df["left_knee_y"],
                                        df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"] = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                        df["right_shoulder_x"], df["right_shoulder_y"])

avg = df[["hip_to_shoulder", "knee_to_hip", "hip_to_hip", 
          "knee_to_ankle", "shoulder_to_shoulder"]].mean()

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key]
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")

X scale: 483.64 cm per unit
Y scale: 241.97 cm per unit
Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.7          0.7        1.4%
knee_to_hip                     44.0            39.7          4.3        9.7%
hip_to_hip                      25.0            25.0          0.0        0.0%
knee_to_ankle                   43.0            35.5          7.5       17.5%
shoulder_to_shoulder            38.0            42.4          4.4       11.5%


In [7]:
import numpy as np

# Load your mediapipe CSV

df = pd.read_csv("../Jakob_movement_CSV/upphöjd-still_mediapipe.csv")


FRAME_WIDTH = 1620  
FRAME_HEIGHT = 1080  

# Convert normalized coords to pixels
for joint in ["head", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
              "left_hand", "right_hand", "left_hip", "right_hip",
              "left_knee", "right_knee", "left_foot", "right_foot"]:
    df[f"{joint}_x"] = df[f"{joint}_x"] * FRAME_WIDTH
    df[f"{joint}_y"] = df[f"{joint}_y"] * FRAME_HEIGHT


def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

# Calculate distances per frame
df["hip_to_shoulder"]        = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["left_shoulder_x"], df["left_shoulder_y"])
df["knee_to_hip"]            = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_hip_x"], df["left_hip_y"])
df["hip_to_hip"]             = euclidean(df["left_hip_x"], df["left_hip_y"],
                                    df["right_hip_x"], df["right_hip_y"])
df["knee_to_ankle"]          = euclidean(df["left_knee_x"], df["left_knee_y"],
                                    df["left_foot_x"], df["left_foot_y"])
df["shoulder_to_shoulder"]   = euclidean(df["left_shoulder_x"], df["left_shoulder_y"],
                                    df["right_shoulder_x"], df["right_shoulder_y"])

# Average across frames
avg = df[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip",
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].mean()

std_devs = df[[
    "hip_to_shoulder",
    "knee_to_hip",
    "hip_to_hip", 
    "knee_to_ankle",
    "shoulder_to_shoulder"
]].std()

print("\nConsistency Check (lower std = more stable):")
for key in ground_truth.keys():
    print(f"{key}: ±{std_devs[key]:.2f} cm")

if std_devs.mean() > 2.0:
    print("\n⚠️ Warning: High variation detected. Person may have moved during recording.")
    
# Ground truth
ground_truth = {
    "hip_to_shoulder":      50,
    "knee_to_hip":          44,
    "hip_to_hip":           25,
    "knee_to_ankle":        43,
    "shoulder_to_shoulder": 38
}


# Scale using hip_to_shoulder as reference
scale = ground_truth["hip_to_shoulder"] / avg["hip_to_shoulder"]
print(f"Scale factor: {scale:.2f} cm per unit\n")

# Print comparison table
print(f"{'Limb':<25} {'Tape (cm)':>10} {'MediaPipe (cm)':>15} {'Error (cm)':>12} {'Error (%)':>10}")
print("-" * 75)
for key, true_val in ground_truth.items():
    mp_val = avg[key] * scale
    error = abs(mp_val - true_val)
    error_pct = error / true_val * 100
    print(f"{key:<25} {true_val:>10.1f} {mp_val:>15.1f} {error:>12.1f} {error_pct:>10.1f}%")


Consistency Check (lower std = more stable):
hip_to_shoulder: ±0.90 cm
knee_to_hip: ±0.61 cm
hip_to_hip: ±0.47 cm
knee_to_ankle: ±1.11 cm
shoulder_to_shoulder: ±0.47 cm
Scale factor: 0.22 cm per unit

Limb                       Tape (cm)  MediaPipe (cm)   Error (cm)  Error (%)
---------------------------------------------------------------------------
hip_to_shoulder                 50.0            50.0          0.0        0.0%
knee_to_hip                     44.0            39.4          4.6       10.5%
hip_to_hip                      25.0            18.6          6.4       25.6%
knee_to_ankle                   43.0            35.2          7.8       18.2%
shoulder_to_shoulder            38.0            31.5          6.5       17.0%


## Data augmentation 

* mirror the movement
* rotate it around y dimension
* add some extra noise to x, y and z

In [8]:
def mirror_sequences_tensor(X, Y):
    """
    Mirror sequences using tensor operations (FAST!)
    
    Args:
        X: (n_sequences, seq_length, 26) - xy coordinates
        Y: (n_sequences, seq_length, 13) - z coordinates
    
    Returns:
        X_mirrored, Y_mirrored
    """
    # Joint indices (0=head, 1=left_shoulder, 2=left_elbow, 3=right_shoulder, ...)
    mirror_pairs = [
        (1, 3),   # shoulders
        (2, 4),   # elbows
        (5, 6),   # hands
        (7, 8),   # hips
        (9, 10),  # knees
        (11, 12), # feet
    ]
    
    X_mirrored = X.clone()
    Y_mirrored = Y.clone()
    
    for left, right in mirror_pairs:
        # Mirror X coordinates (features at indices 2*left and 2*right)
        left_x = 2 * left
        right_x = 2 * right
        
        # Swap and negate x (mirror effect)
        X_mirrored[:, :, left_x], X_mirrored[:, :, right_x] = \
            -X_mirrored[:, :, right_x], -X_mirrored[:, :, left_x]
        
        # Mirror Y coordinates (features at indices 2*left+1 and 2*right+1)
        left_y = 2 * left + 1
        right_y = 2 * right + 1
        
        # Swap y (no negation needed)
        X_mirrored[:, :, left_y], X_mirrored[:, :, right_y] = \
            X_mirrored[:, :, right_y], X_mirrored[:, :, left_y]
        
        # Mirror Z coordinates (direct joint indices)
        Y_mirrored[:, :, left], Y_mirrored[:, :, right] = \
            Y_mirrored[:, :, right], Y_mirrored[:, :, left]
    
    return X_mirrored, Y_mirrored


def rotate_y_sequences_tensor(X, Y, angle_deg):
    """
    Rotate sequences around Y axis using tensor operations
    
    Args:
        X: (n_sequences, seq_length, 26) - xy coordinates
        Y: (n_sequences, seq_length, 13) - z coordinates
        angle_deg: rotation angle in degrees
    
    Returns:
        X_rotated, Y_rotated
    """
    theta = torch.tensor(np.radians(angle_deg), dtype=torch.float32)
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)
    
    X_rotated = X.clone()
    Y_rotated = Y.clone()  # Z coordinates change with rotation
    
    # For each joint (13 joints)
    for joint_idx in range(13):
        # Get x and z coordinates
        x_idx = 2 * joint_idx
        z_idx = joint_idx  # Z is in Y tensor
        
        x = X[:, :, x_idx]
        z = Y[:, :, z_idx]
        
        # Rotate x and z
        X_rotated[:, :, x_idx] = x * cos_t + z * sin_t
        Y_rotated[:, :, z_idx] = -x * sin_t + z * cos_t
    
    return X_rotated, Y_rotated


def add_noise_to_sequences_tensor(X, Y, noise_std_xy=0.002, noise_std_z=0.001, 
                                   apply_to_z=True, prob=0.5, seed=None):
    """
    Add noise to sequences using tensor operations
    
    Args:
        X: (n_sequences, seq_length, 26) - xy coordinates
        Y: (n_sequences, seq_length, 13) - z coordinates
        noise_std_xy: standard deviation for x,y noise
        noise_std_z: standard deviation for z noise
        apply_to_z: whether to add noise to z coordinates
        prob: probability of adding noise to each sequence
        seed: random seed for reproducibility
    
    Returns:
        X_noisy, Y_noisy
    """
    if seed is not None:
        torch.manual_seed(seed)
    
    X_noisy = X.clone()
    Y_noisy = Y.clone()
    
    # Randomly select which sequences to add noise to
    mask = torch.rand(len(X)) < prob
    
    if mask.any():
        n_noisy = mask.sum().item()
        
        # Add noise to X and Y coordinates
        noise_xy = torch.randn(n_noisy, X.shape[1], 26) * noise_std_xy
        X_noisy[mask] = X_noisy[mask] + noise_xy
        
        if apply_to_z:
            noise_z = torch.randn(n_noisy, Y.shape[1], 13) * noise_std_z
            Y_noisy[mask] = Y_noisy[mask] + noise_z
    
    return X_noisy, Y_noisy


In [9]:
def split_data_timeseries(datafolder, seq_length=30, stride=1):
    """
    Load CSV files and create sliding window sequences
    
    Args:
        datafolder: path to CSV files
        seq_length: number of consecutive frames per sequence
        stride: step size between sequences (1 = overlapping, seq_length = non-overlapping)
    
    Returns:
        X: tensor of shape (n_sequences, seq_length, 26)
        y: tensor of shape (n_sequences, seq_length, 13)
    """
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(os.path.join(datafolder, f))
    
    all_X = []
    all_y = []
    
    for file in csv_files:
        df = pd.read_csv(file)
        X, y = input_target_split(df)
        
        # Convert to numpy/tensor for faster processing
        X_np = X.values.astype(np.float32)
        y_np = y.values.astype(np.float32)
        
        # Create sliding window sequences
        n_frames = len(df)
        if n_frames >= seq_length:
            for i in range(0, n_frames - seq_length + 1, stride):
                X_seq = X_np[i:i + seq_length]  # (seq_length, 26)
                y_seq = y_np[i:i + seq_length]  # (seq_length, 13)
                
                all_X.append(torch.tensor(X_seq, dtype=torch.float32))
                all_y.append(torch.tensor(y_seq, dtype=torch.float32))
    
    # Stack all sequences
    all_X = torch.stack(all_X)  # (n_sequences, seq_length, 26)
    all_y = torch.stack(all_y)  # (n_sequences, seq_length, 13)
    
    print(f"Created {len(all_X)} sequences of length {seq_length}")
    print(f"X shape: {all_X.shape}, y shape: {all_y.shape}")
    
    return all_X, all_y

def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data


def split_data_by_video(datafolder, seq_length=30, stride=30, 
                        train_ratio=0.7, val_ratio=0.15, random_seed=42):
    """
    Split by VIDEO FILE to prevent ANY data leakage between train/val/test
    """
    import glob
    
    # Get all CSV files
    csv_files = glob.glob(os.path.join(datafolder, "*.csv"))
    print(f"Found {len(csv_files)} video files")
    
    # Split by FILE, not by sequence
    random.seed(random_seed)
    random.shuffle(csv_files)
    
    n_files = len(csv_files)
    n_train = int(n_files * train_ratio)
    n_val = int(n_files * val_ratio)
    
    train_files = csv_files[:n_train]
    val_files = csv_files[n_train:n_train+n_val]
    test_files = csv_files[n_train+n_val:]
    
    print(f"\nTrain videos: {len(train_files)}")
    print(f"Val videos: {len(val_files)}")
    print(f"Test videos: {len(test_files)}")
    
    def extract_sequences_from_files(file_list):
        """Extract sequences from a list of video files"""
        all_X, all_y = [], []
        
        for file_path in file_list:
            df = pd.read_csv(file_path)
            X_df, y_df = input_target_split(df)
            
            X_np = X_df.values.astype(np.float32)
            y_np = y_df.values.astype(np.float32)
            
            n_frames = len(df)
            # Create sequences from this video
            for i in range(0, n_frames - seq_length + 1, stride):
                X_seq = X_np[i:i + seq_length]
                y_seq = y_np[i:i + seq_length]
                
                all_X.append(torch.tensor(X_seq, dtype=torch.float32))
                all_y.append(torch.tensor(y_seq, dtype=torch.float32))
        
        if not all_X:
            return torch.empty(0, seq_length, 26), torch.empty(0, seq_length, 13)
        
        return torch.stack(all_X), torch.stack(all_y)
    
    # Extract sequences from each split
    train_x, train_y = extract_sequences_from_files(train_files)
    val_x, val_y = extract_sequences_from_files(val_files)
    test_x, test_y = extract_sequences_from_files(test_files)
    
    print(f"\nTrain sequences: {len(train_x)}")
    print(f"Val sequences: {len(val_x)}")
    print(f"Test sequences: {len(test_x)}")
    print(f"X shape: {train_x.shape}")
    print(f"Y shape: {train_y.shape}")
    
    return train_x, train_y, val_x, val_y, test_x, test_y


In [10]:
class zPredictor_timeseries(nn.Module):
    def __init__(self, hidden_layers: list, layer_type="LSTM", dropout=0.0, verbose=True):
        super().__init__()
        self.layer_type = layer_type
        
        input_size = 26  # 13 joints x 2 (x, y)
        
        rnn_class = nn.LSTM if layer_type == "LSTM" else nn.GRU
        self.rnns = nn.ModuleList()
        self.drops = nn.ModuleList()
        
        sizes = [input_size] + hidden_layers
        total_params = 0
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Model: {layer_type} with hidden layers: {hidden_layers}")
            print(f"{'='*60}")
            print(f"{'Layer':<12} {'Input→Hidden':<15} {'Parameters':>20}")
            print("-" * 50)
        
        for i in range(len(hidden_layers)):
            in_size = sizes[i]
            out_size = sizes[i+1]
            
            # Calculate parameters for this layer
            if layer_type == "LSTM":
                # LSTM: 4 * ( (in + out) * out + out )
                layer_params = 4 * ((in_size + out_size) * out_size + out_size)
            else:  # GRU
                # GRU: 3 * ( (in + out) * out + out )
                layer_params = 3 * ((in_size + out_size) * out_size + out_size)
            
            total_params += layer_params
            
            if verbose:
                print(f"L{i+1:<8} {in_size:>3} → {out_size:<4} {layer_params:>20,}")
            
            self.rnns.append(rnn_class(in_size, out_size, batch_first=True))
            self.drops.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
        
        # FC layer
        fc_in = hidden_layers[-1]
        fc_out = 13
        fc_params = (fc_in * fc_out) + fc_out
        total_params += fc_params
        
        if verbose:
            print(f"{'FC':<12} {fc_in:>3} → {fc_out:<4} {fc_params:>20,}")
            print("-" * 50)
            print(f"{'Total':<12} {'':<15} {total_params:>20,}")
            print(f"{'='*60}\n")
        
        self.fc_out = nn.Linear(hidden_layers[-1], 13)
    
    def forward(self, x):
        if self.layer_type == "Dense":
            return self.network(x)
        elif self.layer_type in ("LSTM", "GRU"):
            for rnn, drop in zip(self.rnns, self.drops):
                x, _ = rnn(x)
                x = drop(x)
            return self.fc_out(x)

In [19]:
# Parameters
params = {
    "hidden_layers": [256, 128, 128, 64],
    "layer_type": "LSTM",
    "dropout": 0.45,
    "learning_rate": 0.0007,
    "batch_size": 32,  # Reduced from 64
    "epochs": 2000,
    "run_name": "test_good_settings_augmented",
    "seq_length": 45,
    "stride": 30,
    "weight_decay": 2e-5,  
    #"patience": 5
}

  
random_seed = 42
# Load data with sequences
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"


train_x, train_y, val_x, val_y, test_x, test_y = split_data_by_video(
    datafolder,
    seq_length=params["seq_length"],
    stride=params["stride"],  # Can be 1, 15, or 30 now!
    random_seed=42
)

# Apply mirror augmentation to training data
print(f"Original training size: {len(train_x)}")
train_x_mirror, train_y_mirror = mirror_sequences_tensor(train_x, train_y)

# Concatenate original and mirrored
train_x = torch.cat([train_x, train_x_mirror], dim=0)
train_y = torch.cat([train_y, train_y_mirror], dim=0)

# Shuffle
indices = torch.randperm(len(train_x))
train_x = train_x[indices]
train_y = train_y[indices]

print(f"After mirror augmentation: {len(train_x)}")


print(f"Train sequences: {len(train_x)}")
print(f"Val sequences: {len(val_x)}")
print(f"Test sequences: {len(test_x)}")


# Create DataLoaders directly from tensors
train_dataset = TensorDataset(train_x, train_y)
val_dataset = TensorDataset(val_x, val_y)
test_dataset = TensorDataset(test_x, test_y)


train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

model = zPredictor_timeseries(hidden_layers=params["hidden_layers"], verbose=True, layer_type=params["layer_type"], dropout=params["dropout"]).to(device)

#loss_fn = nn.MSELoss()
#loss_fn = nn.L1Loss()
loss_fn = nn.SmoothL1Loss(beta=0.01)

optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"])

Found 179 video files

Train videos: 125
Val videos: 26
Test videos: 28

Train sequences: 445
Val sequences: 97
Test sequences: 80
X shape: torch.Size([445, 45, 26])
Y shape: torch.Size([445, 45, 13])
Original training size: 445
After mirror augmentation: 890
Train sequences: 890
Val sequences: 97
Test sequences: 80

Model: LSTM with hidden layers: [256, 128, 128, 64]
Layer        Input→Hidden              Parameters
--------------------------------------------------
L1         26 → 256               289,792
L2        256 → 128               197,120
L3        128 → 128               131,584
L4        128 → 64                 49,408
FC            64 → 13                    845
--------------------------------------------------
Total                                     668,749



In [20]:
with mlflow.start_run(run_name=params["run_name"]) as run:
    mlflow.log_params(params)
    
    # Create DataLoaders (only once, inside the run)
    train_dataset = TensorDataset(train_x, train_y)
    val_dataset = TensorDataset(val_x, val_y)
    test_dataset = TensorDataset(test_x, test_y)
    
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(params["epochs"]):
        # Train step with batching
        model.train()
        train_losses = []
        train_maes = []
        train_biases = []
        
        for batch_x, batch_y in train_loader:
            # Move data to device
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = loss_fn(predictions, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Collect batch metrics
            train_losses.append(loss.item())
            train_maes.append(torch.mean(torch.abs(predictions - batch_y)).item())
            train_biases.append(torch.mean(predictions - batch_y).item())
        
        # Average training metrics
        avg_train_loss = np.mean(train_losses)
        avg_train_mae = np.mean(train_maes)
        avg_train_bias = np.mean(train_biases)

        # Evaluation step with batching
        model.eval()
        val_losses = []
        val_maes = []
        val_biases = []
        all_val_preds = []
        all_val_true = []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                val_predictions = model(batch_x)
                loss = loss_fn(val_predictions, batch_y)
                
                val_losses.append(loss.item())
                val_maes.append(torch.mean(torch.abs(val_predictions - batch_y)).item())
                val_biases.append(torch.mean(val_predictions - batch_y).item())
                
                # Store for R² calculation
                all_val_preds.append(val_predictions.cpu().numpy().flatten())
                all_val_true.append(batch_y.cpu().numpy().flatten())
        
        # Average validation metrics
        avg_val_loss = np.mean(val_losses)
        avg_val_mae = np.mean(val_maes)
        avg_val_bias = np.mean(val_biases)
        
        # Calculate R² on all validation data
        val_preds_flat = np.concatenate(all_val_preds)
        val_true_flat = np.concatenate(all_val_true)
        val_r2 = r2_score(val_true_flat, val_preds_flat)
        
        # Calculate training R² (using full training data)
        with torch.no_grad():
            train_predictions_full = model(train_x.to(device))  # Fix: move to device
            train_r2 = r2_score(
                train_y.cpu().numpy().flatten(), 
                train_predictions_full.cpu().numpy().flatten()
            )

        # Log per epoch metrics with step
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "train_mae": avg_train_mae,
            "train_r2": train_r2,
            "train_bias": avg_train_bias,
            "val_loss": avg_val_loss,
            "val_mae": avg_val_mae,
            "val_r2": val_r2,
            "val_bias": avg_val_bias
        }, step=epoch)

        if epoch % 10 == 0:  # Print every 10 epochs to reduce output
            print(f"Epoch {epoch+1}/{params['epochs']} "
                  f"train_loss: {avg_train_loss:.4f} "
                  f"train_mae: {avg_train_mae:.4f} "
                  f"val_loss: {avg_val_loss:.4f} "
                  f"val_mae: {avg_val_mae:.4f} cm")

        # Early stopping
        if params.get("patience"):
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                # Save best model checkpoint
                torch.save(model.state_dict(), 'best_model.pt')
                mlflow.log_artifact('best_model.pt')
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= params["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Test evaluation with batching
    model.eval()
    test_losses = []
    test_maes = []
    test_biases = []
    all_test_preds = []
    all_test_true = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            test_predictions = model(batch_x)
            loss = loss_fn(test_predictions, batch_y)
            
            test_losses.append(loss.item())
            test_maes.append(torch.mean(torch.abs(test_predictions - batch_y)).item())
            test_biases.append(torch.mean(test_predictions - batch_y).item())
            
            all_test_preds.append(test_predictions.cpu().numpy().flatten())
            all_test_true.append(batch_y.cpu().numpy().flatten())
    
    # Average test metrics
    avg_test_loss = np.mean(test_losses)
    avg_test_mae = np.mean(test_maes)
    avg_test_bias = np.mean(test_biases)
    
    # Calculate test R²
    test_preds_flat = np.concatenate(all_test_preds)
    test_true_flat = np.concatenate(all_test_true)
    test_r2 = r2_score(test_true_flat, test_preds_flat)

    mlflow.log_metrics({
        "test_loss": avg_test_loss,
        "test_mae": avg_test_mae,
        "test_r2": test_r2,
        "test_bias": avg_test_bias,  
    })

    # Log the final model
    mlflow.pytorch.log_model(model, artifact_path=f"final_model_{params['run_name']}")
    
    print(f"\n✅ Test Results - Loss: {avg_test_loss:.4f}, MAE: {avg_test_mae:.4f}, R²: {test_r2:.4f}")

2026/04/21 22:49:15 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/21 22:49:15 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/2000 train_loss: 0.1121 train_mae: 0.1170 val_loss: 0.0655 val_mae: 0.0703 cm
Epoch 11/2000 train_loss: 0.0558 train_mae: 0.0606 val_loss: 0.0473 val_mae: 0.0520 cm
Epoch 21/2000 train_loss: 0.0502 train_mae: 0.0550 val_loss: 0.0451 val_mae: 0.0498 cm
Epoch 31/2000 train_loss: 0.0477 train_mae: 0.0525 val_loss: 0.0434 val_mae: 0.0482 cm
Epoch 41/2000 train_loss: 0.0451 train_mae: 0.0498 val_loss: 0.0442 val_mae: 0.0489 cm
Epoch 51/2000 train_loss: 0.0432 train_mae: 0.0480 val_loss: 0.0436 val_mae: 0.0483 cm
Epoch 61/2000 train_loss: 0.0411 train_mae: 0.0458 val_loss: 0.0474 val_mae: 0.0521 cm
Epoch 71/2000 train_loss: 0.0397 train_mae: 0.0444 val_loss: 0.0465 val_mae: 0.0512 cm
Epoch 81/2000 train_loss: 0.0384 train_mae: 0.0431 val_loss: 0.0518 val_mae: 0.0565 cm
Epoch 91/2000 train_loss: 0.0373 train_mae: 0.0420 val_loss: 0.0463 val_mae: 0.0510 cm
Epoch 101/2000 train_loss: 0.0359 train_mae: 0.0406 val_loss: 0.0485 val_mae: 0.0532 cm
Epoch 111/2000 train_loss: 0.0349 train_mae

2026/04/21 23:00:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 23:00:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/04/21 23:00:42 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/21 23:00:42 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



✅ Test Results - Loss: 0.0393, MAE: 0.0440, R²: 0.7858
🏃 View run test_good_settings_augmented at: http://127.0.0.1:5000/#/experiments/3/runs/dff964a5e26e40aab7b65ffac4920ee5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
